In [1]:
cd ..

/home/karaman/Documents/bet


In [2]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
#from src.ingestion.churn_label_generator import generate_churn_labels



In [3]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
players_bronze = spark.read.parquet("./data/bronze/players")
sessions_bronze = spark.read.parquet("./data/bronze/sessions")
transactions_bronze = spark.read.parquet("./data/bronze/transactions")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/01 21:03:41 WARN Utils: Your hostname, karaman-VMware-Virtual-Platform, resolves to a loopback address: 127.0.1.1; using 192.168.1.179 instead (on interface ens33)
26/03/01 21:03:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 21:03:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/01 21:03:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
transactions_bronze.show(4)

+--------------------+---------+--------------------+----------------+------+------------+
|      transaction_id|player_id|      transaction_ts|transaction_type|amount|success_flag|
+--------------------+---------+--------------------+----------------+------+------------+
|4e266d86-908b-407...|   P50000|2024-04-06 10:27:...|         deposit|179.25|        true|
|c2f05dba-0f98-4a4...|   P50002|2024-04-08 21:19:...|         deposit|  8.92|       false|
|d94a6eea-7f4c-4e6...|   P50003|2024-04-10 09:07:...|         deposit| 69.11|        true|
|9fee3671-3fa0-4b6...|   P50005|2024-05-07 00:10:...|      withdrawal| 60.27|        true|
+--------------------+---------+--------------------+----------------+------+------------+
only showing top 4 rows


In [30]:
transactions_bronze.filter(F.col('success_flag')==False).count()


4073

In [31]:
transactions_bronze = (
    transactions_bronze
    .withColumn(
        "signed_amount", 
        F.when(F.col("transaction_type") == "deposit", F.col("amount") * F.col("success_flag").cast('int'))
         .when(F.col("transaction_type") == "withdrawal", -F.col("amount") * F.col("success_flag").cast('int'))
         .otherwise(F.lit(0.0))

    )
)